# Sessions 14–15: Sequence Models and Transformer Applications

**Learning goals:**

- Trace raw text through tokenization, embeddings, contextual representations and task outputs.
- Explain RNN and LSTM state, attention and Transformer information flow using examples.
- Distinguish sequence-to-sequence generation from sequence classification.
- Compare encoder–decoder, encoder-only and decoder-only language models.
- Implement feature engineering + MLP, BERT and GPT classifiers on the same tickets.
- Compare held-out results, analyse mistakes and save models for reuse.

**Prerequisites:**

- Neural networks, PyTorch, training, evaluation and transfer learning.
- Basic Python functions, lists, dictionaries and tensor shapes.
- Internet for initial model and dataset downloads.



## Outline

1. From CNNs to sequence models
2. Tokenization and embedding
3. RNN and LSTM: a brief introduction
4. Attention and Transformers
5. Large language models for natural language processing
6. Case study: support ticket classification


## 1. From CNNs to Sequence Models

Deep learning learns useful **feature representations** from raw data, then uses them to produce a task output. “Deep” refers to multiple layers that progressively transform and combine information; training adjusts these layers and the output head together.

**Raw input → learned feature representations → task output**

For an image, early CNN layers can detect edges and simple patterns. Later layers combine them into more complex shapes and object features. A classification head uses these features to predict a class.

![CNN: image to tensor, learned spatial features, classification head and output.](images/cnn-image2.png)


**Sequence data** includes text, speech and time-series measurements. Its elements are ordered and often depend on one another:

- **Order matters:** “The dog chased the cat” differs from “The cat chased the dog”.
- **Context matters:** “bank” means something different beside “loan” and “river”.
- **Length varies:** sentences contain different numbers of tokens, and a translation can differ in length from its source.

Sequence models learn features from these relationships. In ticket classification, a Transformer combines information across the text, and a classification head predicts the ticket category.

![Sequence model: text to token IDs, embeddings, contextual features, classification head and output.](images/sequence-image2.png)



## 2. Tokenization and Embedding

Text needs two conversions before the network processes it: **tokenization** splits text into tokens and assigns IDs; **embedding** turns those IDs into learned numerical vectors. Transformer layers then combine information across positions to build contextual features.

![Text passes through a tokenizer into IDs, embedding vectors and Transformer layers to produce contextual features.](images/text-model-overview-image2.png)


### 2.1 Tokenization

A **tokenizer** splits text into units such as words, subword pieces or characters, then looks up each unit in its vocabulary. Tokens can have different granularities:

- **Character-level:** each token represents a character.
- **Word-level:** each token represents a whole word.
- **Subword-level:** each token represents a subword piece; a word may use one or several pieces.

For example, an illustrative split of `cats` is `c | a | t | s` at character level, `cats` at word level, or `cat | s` at subword level. Actual subword boundaries depend on the tokenizer.

For the toy vocabulary below, `Cats sleep.` becomes `[12, 47, 3]`.

![A toy tokenizer splits Cats sleep. into pieces and maps them to vocabulary IDs.](images/tokenization-image2.png)

**The tokenizer is paired with the model.** Models using different tokenizers can produce different token boundaries, IDs and token counts for the same text. Some models share a tokenizer, so their results can match. Always use the tokenizer supplied for your model: an ID must select the embedding row that the model learned for that token.

**Explore:** Open the [OpenAI Tokenizer](https://platform.openai.com/tokenizer) and enter `An unpredictable Wi-Fi problem!`. Inspect the pieces and token count. Where model or encoding choices are available, compare the same text across them; try adding a space or changing a word. Do not assume a token equals one word or character.


### 2.2 Embedding

**Purpose:** token IDs identify vocabulary entries but do not describe their meaning. Embeddings give the network compact numerical features it can learn to use. Unlike a long, sparse one-hot vector, an embedding contains a smaller set of continuous values.

**Method:** embedding converts each token ID into a fixed-dimensional vector. The model stores an **embedding table**, with one vector per vocabulary entry. These vectors are usually **model parameters learned through training**: embedding gives each token a **learnable vector**. Each token ID selects its row. The selected vectors stay in input order: three tokens with four values each form a `3 × 4` matrix.

![Token IDs select learned vectors in input order from an embedding table.](images/embedding-image2.png)

The table values are learned during pretraining and can change during fine-tuning. Training can give related tokens useful shared patterns; nearby ID numbers do not imply similar meanings, and vector dimensions have no fixed human labels.

Embedding is the starting representation. For the same `bank` token ID, the initial embedding is identical in `river bank` and `bank loan`. After Transformer layers combine context and position information, the **contextual representations differ**, reflecting the different surrounding words. These contextual features support classification or generation.

[Video explanation: embeddings](https://www.youtube.com/watch?v=OATCgQtNX2o)


{{< video https://www.youtube.com/embed/OATCgQtNX2o?hl=en title="Embeddings explained" aria-label="Video explanation of embeddings" aspect-ratio="16x9" >}}


## 3. RNN and LSTM: A Brief Introduction

A **Recurrent Neural Network (RNN)** reads a sequence one step at a time. Each step combines the current input with the previous hidden state to produce a new representation. The same parameters are reused across steps.

A **Long Short-Term Memory (LSTM)** network adds a memory pathway and gates to control what information is kept, updated and exposed. This helps it learn longer dependencies.


### A Character-Level RNN Example

![An RNN reads h, e, l, l and scores the next characters e, l, l, o.](images/rnn-hell-forward-pass.png)

Read the figure from left to right:

- **Input:** `h, e, l, l`, encoded with four one-hot values for vocabulary `h, e, l, o`.
- **Hidden state:** three units carry context between steps.
- **Output:** four next-character scores; targets are `e, l, l, o`. Training raises green target scores relative to red alternatives. These are raw scores, not probabilities.

The two `l` inputs have different targets: the hidden state lets the model use their different contexts.

[Karpathy: The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) · [Awesome RNN resources](https://github.com/kjw0612/awesome-rnn)

### LSTM: Managing Memory Across Time

![Two LSTM steps use gates to update cell and hidden states.](images/lstm-memory-gates-image2.png)

The **cell state** is the upper memory pathway. The **hidden state** is the current output representation passed to the next step.

| Gate | Role |
|---|---|
| Forget | Retain useful old memory. |
| Input | Write selected new information. |
| Output | Control how much updated memory contributes to the hidden state. |

Gates use continuous values between 0 and 1. For “I grew up in France … I speak fluent French”, useful information about France can be retained across intervening words. Memory remains limited, and processing is still sequential.

[Hochreiter and Schmidhuber: Long Short-Term Memory (1997)](https://doi.org/10.1162/neco.1997.9.8.1735)


Imagine translating a long notice containing a date, room number and several instructions. A single final summary must support all output decisions. This fixed-size bottleneck motivates giving the decoder access to representations at all input positions.



## 4. Attention and Transformers

### 4.1 Attention

**Attention builds a useful feature vector by giving more weight to relevant information and less weight to other information.** In RNN/LSTM Seq2Seq models, it lets the decoder access different input positions rather than depend only on a final summary.

**Input → output:**

| Input | Role |
|---|---|
| Query (Q) | The current representation requesting information. |
| Keys (K) | Representations compared with the query to calculate weights. |
| Values (V) | Information combined using those weights. |

**Query + Keys → attention weights → weighted combination of Values → output feature vector.** Each query produces an output feature vector. Attention does not directly output a translated word; later decoder processing and the vocabulary output layer predict it.

![Attention principle: a query is compared with keys to calculate weights; the weights combine values into an output feature vector.](images/attention-principle-image2.png)

**Read the attention diagram:** thicker arrows show a larger contribution from `nine`. The combined feature vector helps later layers predict `neuf` (“nine”). The lower flow shows how this happens: compare Q with K to calculate weights, then use those weights to combine V. The illustrated weights are not measurements from a trained model.

**Example: different output steps use different source weights**

![](images/attention-translation-example-image2.png)

Read each row from left to right. When predicting `J’aime`, the query gives weights `0.45, 0.50, 0.05` to the source positions `I, like, tea`. When predicting `thé`, the weights become `0.05, 0.05, 0.90`, focusing more strongly on `tea`. Each row sums to 1 and produces its own context vector by combining source value vectors.

The source representations stay available, but the decoder query and weights change with the generated prefix. Decoder processing and the output head use the context vector to help predict the next word. This is a word-level illustration with invented weights; an actual model operates on its tokenizer’s tokens.

[Original Transformer paper](https://arxiv.org/abs/1706.03762) · [3Blue1Brown lesson](https://www.3blue1brown.com/lessons/attention/)

[Video explanation: attention examples and attention patterns (1:39–11:08)](https://www.youtube.com/watch?v=eMlx5fFNoYc&t=99s)


{{< video https://www.youtube.com/embed/eMlx5fFNoYc?hl=en&start=99&end=668 start="99" title="Attention examples and attention patterns, 1:39–11:08" aria-label="3Blue1Brown attention explanation from 1 minute 39 seconds to 11 minutes 8 seconds" aspect-ratio="16x9" >}}


**Multi-head attention**

![Several attention heads learn separate projections and weights; their output vectors are concatenated and projected to produce updated representations.](images/multihead-attention-image2.png)

The input representations enter several heads, each with its own learned projections and weights. Their output vectors are concatenated and projected to form updated features. Read the diagram from left to right: input → parallel heads → concatenate and project → updated representations.

In **self-attention**, Q, K and V come from the same sequence. In **cross-attention**, Q comes from the decoder while K and V come from encoder outputs, connecting output generation to the source.

[Translation attention](https://arxiv.org/abs/1409.0473) · [Multi-head attention](https://arxiv.org/abs/1706.03762)

[Video explanation: multiple heads (19:19–22:16)](https://www.youtube.com/watch?v=eMlx5fFNoYc&t=1159s)


{{< video https://www.youtube.com/embed/eMlx5fFNoYc?hl=en&start=1159&end=1336 start="1159" title="Multiple attention heads, 19:19–22:16" aria-label="3Blue1Brown multiple attention heads explanation from 19 minutes 19 seconds to 22 minutes 16 seconds" aspect-ratio="16x9" >}}


### 4.2 Transformer

The original **Transformer** has three key structural features:

- **Seq2Seq structure:** an encoder represents the source sequence; a decoder uses those representations to generate a target sequence.
- **No recurrent state updates:** it removes the step-by-step recurrence used by RNNs/LSTMs, allowing many sequence positions to be calculated in parallel during training.
- **Repeated attention:** stacked blocks use multi-head attention and feed-forward layers to refine features. Position information makes sequence order available.

![](images/encoder-decoder-input-output-image2.png)

| Component | Input | Output |
|---|---|---|
| Encoder | Source token embeddings with position information | A sequence of contextual source features |
| Decoder | Source features and target-prefix embeddings with position information | Contextual target features |
| Vocabulary output head | Decoder features | Scores over possible next tokens |

The decoder therefore uses **two sources of information**: what the input says and what has already been generated. It produces feature vectors; the output head converts them into token scores.

**Example: “I like tea” → “J’aime le thé”**

![](images/encoder-decoder-translation-example-image2.png)

The encoder produces a contextual representation for each source position. At every generation step, the decoder can access those source features and uses an increasingly long target prefix:

| Step | Target prefix | Next output |
|---|---|---|
| 1 | `<start>` | `J’aime` |
| 2 | `<start> J’aime` | `le` |
| 3 | `<start> J’aime le` | `thé` |

The three decoder panels show the **same model reused across steps**, not three separately trained decoders. This word-level example omits the final end token and simplifies tokenization.

![](images/transformer-seq2seq-image2.png)

Encoder self-attention combines source information. Decoder masked self-attention uses the target prefix, and cross-attention accesses encoder features. Feed-forward layers further transform the representations.

**Encoder and decoder: similar blocks, different information access**

Both stacks use attention, feed-forward networks, residual connections and layer normalisation. Their main differences are the information each position can access and whether they use cross-attention:

| Block type | Self-attention access | Cross-attention |
|---|---|---|
| Encoder | All valid source positions, including both left and right context | None |
| Original Transformer decoder | Current and earlier target-prefix positions; future positions are masked | Uses encoder outputs as keys and values, with decoder features as queries |
| GPT-style decoder | Current and earlier positions in the prompt and generated sequence | None; there is no separate encoder |

Cross-attention connects the decoder to the **encoder outputs**, not directly to the raw source tokens. Without cross-attention, a GPT block looks very similar to an encoder block, but its **causal self-attention** preserves the decoder-style information flow needed for next-token prediction. This is why GPT is called **decoder-only**, while BERT uses bidirectional encoder blocks and is called **encoder-only**. The names describe architecture and information access, not where a block sits in a diagram.

**Feed-forward network: an MLP for each token**

The Transformer **feed-forward network (FFN)** is a small **multilayer perceptron (MLP)**: typically two fully connected layers with a nonlinear activation between them.

**Token feature vector → fully connected layer → activation → fully connected layer → updated feature vector**

The first layer usually expands the feature dimension; the second maps it back to the model dimension. Within a block, the same MLP is applied independently to every token position, sharing its parameters across positions. **Attention mixes information between tokens; the FFN further transforms each token’s features without directly mixing positions.**

**Translation at inference:** encode the source, start the decoder with a start token, predict a token, append it to the prefix and repeat until an end token or length limit.

During training, shifted reference targets supply the prefixes and a **causal mask** hides future target tokens. Multiple positions can be calculated in parallel; autoregressive generation still proceeds token by token.


## 5. Large Language Models for Natural Language Processing

Pretrained language models reuse Transformer representations across language tasks. Three common architecture families differ in how they read input and produce output. The examples include both early pretrained models and later large models.

| Architecture | Main role and applications | Examples |
|---|---|---|
| **Encoder–decoder** | Encode a source sequence, then generate a target sequence. Used for translation, summarisation and response generation. | Original Transformer; Google **T5** |
| **Encoder-only** | Build bidirectional contextual representations of input. Add a task head for classification, sentiment analysis, tagging or extractive question answering. | Google **BERT** |
| **Decoder-only** | Extend a prompt by predicting tokens from the available prefix. Used for continuation, dialogue and other prompted generation tasks. | OpenAI **GPT**, Meta **LLaMA/Llama**, **DeepSeek-V3/R1** |

Encoder-only describes the architecture, not one-way reading: BERT uses both left and right context. Decoder-only models have no separate source encoder; the prompt and generated tokens form their working sequence.

### 5.1 Selected Development Milestones

![Selected milestones from Transformer in 2017 to DeepSeek-R1 in 2025, grouped by architecture family.](images/nlp-model-timeline-horizontal-image2.png)

The timeline selects milestones from 2017 to 2025, grouped by architecture family.

- **Attention and parallel training:** Transformer removes recurrence, connects relevant token positions through attention and processes many positions in parallel during training. Autoregressive generation still proceeds token by token. See [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
- **Scaling laws:** increasing model size, training data and computation can predictably reduce language-modelling loss within studied ranges. This is evidence of continued improvement over those ranges, not a guarantee of unlimited improvement or better performance on every task. See [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361).
- **Conversational applications:** [ChatGPT launched on 30 November 2022](https://openai.com/index/chatgpt/). Its conversational interface let users ask follow-up questions and refine requests over several turns, bringing language-model applications to a wider audience. Language models support text generation, information extraction, translation and code generation, though outputs can contain errors.

Timeline sources: [Transformer](https://arxiv.org/abs/1706.03762), [BERT](https://arxiv.org/abs/1810.04805), [T5](https://arxiv.org/abs/1910.10683), [GPT](https://openai.com/index/language-unsupervised/), [GPT-2](https://openai.com/index/better-language-models/), [GPT-3](https://arxiv.org/abs/2005.14165), [LLaMA](https://arxiv.org/abs/2302.13971), [Llama 3](https://ai.meta.com/blog/meta-llama-3/), [DeepSeek-V3](https://arxiv.org/abs/2412.19437), [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

### 5.2 BERT: Representing Input with Both-Sided Context

[BERT paper](https://arxiv.org/abs/1810.04805)

[Video explanation: BERT](https://www.youtube.com/watch?v=AprUD-TSUYE)


{{< video https://www.youtube.com/embed/AprUD-TSUYE?hl=en title="BERT video explanation" aria-label="Video introduction to BERT at the start of Section 5.2" aspect-ratio="16x9" >}}


**BERT** means **Bidirectional Encoder Representations from Transformers**. Its encoder uses left and right context to build a representation for each token.

- **Pretraining:** predict masked tokens; original BERT also learned a next-sentence prediction task.
- **Task use:** attach and fine-tune a task head, such as a ticket-category classifier.
- **Example:** `The bank approved the [MASK].` uses surrounding words to suggest the missing token.

![BERT: token embeddings pass through a bidirectional Transformer encoder; selected contextual features feed a masked-token head or a classification head.](images/bert-architecture-image2.png)

**Read the structure:** token, position and segment embeddings enter a stack of encoder blocks. Each block combines bidirectional self-attention with a feed-forward network, using residual connections and layer normalisation. The masked-token head uses the feature at `[MASK]` to predict a missing token; a classification head uses the `[CLS]` feature to predict a sequence label. The diagram shows alternative task heads. See the [BERT paper](https://arxiv.org/abs/1810.04805).

The ticket practical uses **DistilBERT**, a smaller BERT-family encoder, with a new classification head fine-tuned on labelled tickets.


### 5.3 T5: Text In, Text Out

**T5** means **Text-to-Text Transfer Transformer**. It uses an **encoder–decoder** architecture and expresses tasks as input text mapped to output text.

- **Encoder:** reads the whole input with bidirectional self-attention and produces contextual source features.
- **Decoder:** uses masked self-attention over the target prefix and cross-attention to the encoder features, then predicts the next token through a vocabulary head.
- **Pretraining:** reconstruct spans removed from text, using special sentinel tokens to mark the missing spans.
- **Task use:** fine-tune on input–output text pairs, often with a task prefix such as `translate English to German:` or `summarize:`.

![T5: a bidirectional encoder supplies source features to a decoder through cross-attention; the decoder uses a target prefix to predict the next token.](images/t5-architecture-image2.png)

**Read the structure:** source text enters the encoder on the left. The decoder on the right receives both the target prefix and encoder features; cross-attention connects the two stacks. A vocabulary head converts decoder features into next-token scores. Position information uses relative position biases in attention. Residual connections and layer normalisation support both stacks.

**Example:** `translate English to German: That is good.` → `Das ist gut.` At inference, the decoder starts with its start token and extends the output one token at a time. During training, shifted reference targets and a causal mask allow multiple target positions to be calculated in parallel.

BERT typically adds a task-specific prediction head; T5 generates text, including label words for classification tasks. See the [T5 paper](https://arxiv.org/abs/1910.10683).

### 5.4 GPT: Generating from a Prefix

**GPT** means **Generative Pre-trained Transformer**. A decoder-only model uses causal self-attention: each position can use the available prefix, but not future tokens.

- **Pretraining:** predict the next token from preceding tokens.
- **Generation:** predict a token, append it to the prefix and repeat.
- **Applications:** text continuation, dialogue, summarisation and code generation, depending on the model and its subsequent training.

**Example:** `The support team checked the network and` becomes the prefix for continued text. Prompting changes the input; fine-tuning changes model parameters.

![GPT: embeddings pass through repeated decoder blocks with masked self-attention and feed-forward layers, then a vocabulary head predicts the next token.](images/gpt-architecture-image2.png)

**Read the structure:** token and position embeddings enter repeated decoder blocks. Masked self-attention prevents each position from seeing future tokens; feed-forward layers refine the features. Residual connections and layer normalisation support each block. The vocabulary head predicts the next token; an optional classification head can instead produce task labels. GPT has no separate encoder or encoder–decoder cross-attention. `N` is the number of blocks and depends on the model version. This simplified diagram omits individual normalisation steps. See the [GPT paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf).

GPT-2 is a base continuation model; instruction-following assistants require further training. Generated text can contain repetition or incorrect claims.


## 6. Implementing Three Methods for Support Ticket Classification

### 6.1 Data: Support Tickets

The supplied CSV contains **320 fictional teaching tickets**, with 80 in each category. The fixed splits contain **208 training, 48 validation and 64 test tickets**. The original 80 tickets and their splits are preserved; new examples add varied wording, negation and context. The data is synthetic and balanced, so these results do not estimate real service-desk performance.

| Category | Routing rule | Example |
|---|---|---|
| Account access | Credentials, locked accounts and authentication | My account is locked. |
| Hardware | Physical device faults | The laptop screen is cracked. |
| Software | Application installation, crashes and behaviour | The spreadsheet application crashes. |
| Network | Connectivity and network access | The campus Wi-Fi disconnects. |

A real ticket can involve several issues. This exercise assigns one primary category; ambiguous cases should be reviewed rather than forcing an unjustified label.

[Download the practice notebook](<practice/Session14-15 Support Ticket Classification Practice.ipynb>){download="Session14-15 Support Ticket Classification Practice.ipynb"} · [Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/14-15-Sequence-Models-and-Transformers/practice/Session14-15%20Support%20Ticket%20Classification%20Practice.ipynb)

[Download the ticket CSV](sample_data/support_tickets.csv).


In [ ]:
import pandas as pd

TICKET_CSV = "sample_data/support_tickets.csv"
tickets = pd.read_csv(TICKET_CSV)  # Open the supplied labelled data when first used.
print(tickets.groupby(["split", "category"]).size().unstack(fill_value=0))
print(tickets.head(8).to_string(index=False))
assert not tickets["text"].str.lower().duplicated().any(), "Check duplicate ticket text before training."

class_names = ["Account access", "Hardware", "Software", "Network"]
label_to_id = {name: number for number, name in enumerate(class_names)}
id_to_label = {number: name for name, number in label_to_id.items()}
tickets["label"] = tickets["category"].map(label_to_id)
train_tickets = tickets.loc[tickets["split"] == "train"].copy()
validation_tickets = tickets.loc[tickets["split"] == "validation"].copy()
test_tickets = tickets.loc[tickets["split"] == "test"].copy()


**Choosing the methods**

Implement three approaches on the same labelled tickets:

| Method | Input representation | Classifier |
|---|---|---|
| Feature Engineering + MLP | Fixed word-count features | A scikit-learn MLP trained from scratch |
| BERT | Learned bidirectional contextual features | Fine-tuned DistilBERT with a classification head |
| GPT | Learned causal contextual features | Fine-tuned GPT-2 with a classification head |

All methods share the data splits and label mapping defined above. Use validation data to select weights and reserve test data for the final comparison.



### 6.2 Feature Engineering + MLP

**Ticket text → word-count features → MLP → category.** 

The feature engineering step defines a bag-of-words representation; the MLP learns how those features relate to the labels.

`CountVectorizer` builds a vocabulary from **training text only**. Each ticket becomes a sparse vector of word counts. `MLPClassifier` provides a fully connected hidden layer with ReLU activation and an output layer for the four categories. Its `fit()` method handles optimisation internally, so we do not write a PyTorch training loop.

Try hidden layers with 16 and 32 units, train each on the training split, and select the model with the higher validation macro F1. `random_state` makes each initialisation reproducible. The `lbfgs` solver suits this small dataset. Keep the count matrices sparse; counts still ignore word order and context.

See the [scikit-learn MLPClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

If needed, install the libraries before starting the notebook:

```bash
python -m pip install pandas scikit-learn matplotlib joblib
```


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
count_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 1))
# Fit the vocabulary on training tickets only; reuse it for validation.
train_counts = count_vectorizer.fit_transform(train_tickets["text"])
validation_counts = count_vectorizer.transform(validation_tickets["text"])
y_count_train = train_tickets["label"].to_numpy()
y_count_validation = validation_tickets["label"].to_numpy()

print("Vocabulary size:", len(count_vectorizer.vocabulary_))
print("Training input shape:", train_counts.shape)
print("First ticket:", train_tickets.iloc[0]["text"])
first_counts = train_counts[0].toarray()[0]  # Densify one row only for inspection.
print([(word, int(first_counts[index])) for index, word in enumerate(count_vectorizer.get_feature_names_out()) if first_counts[index] > 0])


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

COUNT_HIDDEN_UNITS = (16, 32)
count_model = None
best_count_macro_f1 = -1.0
count_validation_rows = []

for hidden_units in COUNT_HIDDEN_UNITS:
    candidate = MLPClassifier(
        hidden_layer_sizes=(hidden_units,), activation="relu",
        alpha=0.01, max_iter=1000, random_state=SEED,
    )
    candidate.fit(train_counts, y_count_train)  # Learn from training tickets only.
    validation_predictions = candidate.predict(validation_counts)
    validation_macro_f1 = f1_score(
        y_count_validation, validation_predictions,
        labels=list(id_to_label), average="macro", zero_division=0,
    )
    count_validation_rows.append({
        "hidden_units": hidden_units,
        "accuracy": accuracy_score(y_count_validation, validation_predictions),
        "macro_f1": validation_macro_f1,
    })
    if validation_macro_f1 > best_count_macro_f1:
        best_count_macro_f1 = validation_macro_f1
        count_model = candidate  # Retain the fitted model selected on validation data.

print(pd.DataFrame(count_validation_rows).to_string(index=False))
print("Selected hidden layer:", count_model.hidden_layer_sizes)


### 6.3 BERT: Fine-Tuning DistilBERT

**Ticket text → token IDs → bidirectional encoder → classification head → category.** 

DistilBERT is the smaller BERT-family model used for this implementation. The pretrained checkpoint is [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased). Unlike word counts, its learned features depend on token order and context.

**Loading BERT with Hugging Face**

Hugging Face **Transformers** provides pretrained model and tokenizer classes; **Datasets** organises the ticket splits. The loading code below follows these steps:

1. `Dataset.from_pandas()` and `DatasetDict` convert the shared DataFrames into train, validation and test datasets. They are dataset containers, not tensors.
2. `AutoTokenizer.from_pretrained(BERT_CHECKPOINT)` loads the vocabulary and tokenization rules paired with DistilBERT. Mapping `prepare_tickets()` over the datasets produces token IDs and attention masks.
3. `AutoModelForSequenceClassification.from_pretrained(...)` loads pretrained encoder weights and creates a classification head with `num_labels=4`. `id2label` and `label2id` connect numerical predictions to category names. Warnings about newly initialised head weights are expected.
4. `DataCollatorWithPadding` pads each batch to a common length and converts the examples into tensors. `cache_dir` controls where downloaded files are stored; it does not train the model.

See the [Hugging Face text classification guide](https://huggingface.co/docs/transformers/v4.57.1/en/tasks/sequence_classification).

**Prepare inputs and the task head**

Each ticket label is one integer identifying its category. DistilBERT supplies pretrained language features, and a newly initialised classification head supplies four task scores. The initialisation warning for the new head is expected: its categories have not been learned yet.

Padding makes batch lengths compatible; its attention mask marks real tokens. Truncation limits input length and can remove useful details from long tickets.


Before starting the notebook, install these packages in your terminal; restart the kernel if it is already running. These packages support both the BERT and GPT methods. The practical uses the Transformers 4.57 API. Run from the notebook’s own folder so that relative paths resolve correctly. A GPU is recommended for fine-tuning; GPT-2 requires more memory than DistilBERT.

```bash
python -m pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn pandas matplotlib joblib
```

Model caches, training checkpoints and saved classifiers are stored in the course’s `transformer/` folder, which is ignored by Git. Both notebooks use `transformer/` in the current working directory. The code creates each folder when first needed.

`Trainer` automatically uses CUDA or Apple MPS when available, and CPU otherwise. Each training cell prints its selected device. The scikit-learn MLP uses CPU.


In [ ]:
import os
from datasets import Dataset, DatasetDict

ticket_splits = DatasetDict({
    name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    for name, frame in [("train", train_tickets), ("validation", validation_tickets), ("test", test_tickets)]
})

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed

BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
TICKET_TOKENIZER_CACHE_DIR = "transformer/ticket-tokenizer-cache"
os.makedirs(TICKET_TOKENIZER_CACHE_DIR, exist_ok=True)

# Use the tokenizer paired with the pretrained encoder.
ticket_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT, cache_dir=TICKET_TOKENIZER_CACHE_DIR)
SEED = 42

MAX_TICKET_TOKENS = 96

def prepare_tickets(batch):
    return ticket_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

encoded_tickets = ticket_splits.map(prepare_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Initialise the new task head reproducibly.
TICKET_MODEL_CACHE_DIR = "transformer/ticket-model-cache"
os.makedirs(TICKET_MODEL_CACHE_DIR, exist_ok=True)
ticket_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=len(class_names), cache_dir=TICKET_MODEL_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
)
ticket_collator = DataCollatorWithPadding(tokenizer=ticket_tokenizer)


**Train and validate BERT**

**How Hugging Face trains the classifier**

`TrainingArguments` sets the learning rate, batch sizes, maximum epochs, output directory and evaluation schedule. `Trainer` combines these settings with the model, datasets, tokenizer, collator and metric function.

Calling `ticket_trainer.train()` runs the training loop: create batches, calculate classification loss, backpropagate gradients and update the encoder and classification head. This is **fine-tuning an existing pretrained model**, rather than pretraining DistilBERT from scratch.

The maximum is **12 epochs**. An epoch is one pass through the training data. Evaluate after every epoch and use `EarlyStoppingCallback(early_stopping_patience=3)` to stop after three evaluations without an improvement in validation macro F1. `load_best_model_at_end=True` restores the best validation checkpoint, which may come from an earlier epoch. More training is useful only while validation performance improves. See the [Trainer documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer) and [early stopping documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/callback#transformers.EarlyStoppingCallback).

Accuracy counts all correct predictions. Macro F1 gives each category equal weight. A confusion matrix shows where errors occur.

Compare against a majority-class baseline. The newly initialised head must learn the ticket labels from the labelled examples.

**Optional experiment — freeze the DistilBERT encoder**

Freezing sets `requires_grad=False` for the pretrained encoder parameters. The encoder still processes each ticket, and its weights stay fixed while the `pre_classifier` and `classifier` layers learn the ticket labels. The runnable example below uses full fine-tuning. To try head-only training, use the following code on a freshly loaded model before creating the `Trainer`.

```python
# Freeze the pretrained encoder and inspect the trainable layers.
ticket_model.base_model.requires_grad_(False)
for name, parameter in ticket_model.named_parameters():
    if parameter.requires_grad:
        print(name, parameter.numel())
```

The output should list the weights and biases of `pre_classifier` and `classifier`. Each count is the number of scalar parameters in that tensor.


In [ ]:
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

def ticket_metrics(prediction):
    logits, labels = prediction
    predicted = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, labels=list(id_to_label), average="macro", zero_division=0),
    }

TICKET_CHECKPOINT_DIR = "transformer/ticket-checkpoints"
os.makedirs(TICKET_CHECKPOINT_DIR, exist_ok=True)
BERT_MAX_EPOCHS = 12  # Maximum passes; early stopping may finish sooner.
ticket_args = TrainingArguments(
    output_dir=TICKET_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=BERT_MAX_EPOCHS, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("BERT training device:", ticket_args.device)  # Show the device selected by Trainer.
ticket_trainer = Trainer(
    model=ticket_model, args=ticket_args,
    train_dataset=encoded_tickets["train"], eval_dataset=encoded_tickets["validation"],
    processing_class=ticket_tokenizer, data_collator=ticket_collator, compute_metrics=ticket_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
ticket_trainer.train()
print("Epochs completed:", ticket_trainer.state.epoch)
print("Best checkpoint:", ticket_trainer.state.best_model_checkpoint)
print("Validation:", ticket_trainer.evaluate())


### 6.4 GPT: Fine-Tuning GPT-2

**Ticket text → token IDs → causal decoder → last-token feature → classification head → category.**

This implementation uses the downloadable [GPT-2 checkpoint](https://huggingface.co/openai-community/gpt2). It applies supervised fine-tuning to a decoder-only model, using the same ticket splits and labels as the other methods. A newly initialised classification head produces four category scores; both this head and the pretrained decoder are updated.

GPT-2 was pretrained to predict the next token. Here we use a **sequence classification head**, rather than asking it to generate a label as text. The last non-padding token can attend to all earlier ticket tokens, so its contextual feature can represent the available ticket. There is no separate encoder or cross-attention. This example runs locally and requires no API key.

**Loading GPT with Hugging Face**

The same Hugging Face interfaces select a different architecture from the checkpoint configuration:

1. `AutoTokenizer.from_pretrained(GPT_CHECKPOINT)` loads GPT-2’s tokenizer. Its vocabulary differs from DistilBERT’s, so token IDs must be prepared again with `prepare_gpt_tickets()`.
2. Set the padding token and use the same `pad_token_id` in the tokenizer and model. The attention mask identifies real input positions.
3. `AutoModelForSequenceClassification.from_pretrained(...)` loads the pretrained GPT-2 decoder and adds a four-class prediction head. It selects the final non-padding token’s feature for the sequence prediction. The head needs training; it does not inherit knowledge of our ticket labels.
4. `DataCollatorWithPadding` creates batches for this tokenizer. `use_cache=False` disables the token-generation cache, which is unnecessary for this classification task.

**Prepare inputs and the task head**

GPT-2 has no default padding token. Reuse its end-of-text token for padding, set the model’s `pad_token_id` to match, and keep the attention mask. Right padding leaves the real ticket tokens at their original positions. See the [GPT-2 sequence classification documentation](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/gpt2#transformers.GPT2ForSequenceClassification).


In [ ]:
import os
GPT_CHECKPOINT = "openai-community/gpt2"
GPT_CACHE_DIR = "transformer/gpt-cache"
os.makedirs(GPT_CACHE_DIR, exist_ok=True)
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_CHECKPOINT, cache_dir=GPT_CACHE_DIR)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token  # Reuse an existing token without resizing embeddings.
gpt_tokenizer.padding_side = "right"

def prepare_gpt_tickets(batch):
    return gpt_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

gpt_encoded_tickets = ticket_splits.map(prepare_gpt_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Make the new classification head reproducible.
gpt_model = AutoModelForSequenceClassification.from_pretrained(
    GPT_CHECKPOINT, num_labels=len(class_names), cache_dir=GPT_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
    pad_token_id=gpt_tokenizer.pad_token_id,
)
gpt_model.config.use_cache = False  # Classification does not need a generation cache.
gpt_collator = DataCollatorWithPadding(tokenizer=gpt_tokenizer)


**Train and validate GPT**

Reuse the accuracy and macro F1 function from the BERT method. Use the same token limit, validation split and maximum of **12 epochs**, with a smaller batch size for GPT-2. `EarlyStoppingCallback` stops training after three epochs without improved validation macro F1, and the best validation checkpoint is restored.

**Training GPT with Hugging Face**

`TrainingArguments` configures this run, and `Trainer` receives the GPT-2 model, its tokenized datasets and its own padding collator. Calling `gpt_trainer.train()` updates both the decoder and the classification head using the category labels. The training loop uses classification loss, rather than the next-token loss used in GPT-2 pretraining. `gpt_trainer.evaluate()` then measures validation performance. No API request or API key is needed.

The two Transformer models differ in size and pretraining. Compare frozen and fully fine-tuned runs separately for each model, using the same data splits.

**Optional experiment — freeze the GPT-2 decoder**

For this GPT-2 classifier, `base_model` contains the pretrained decoder and `score` is the classification head. The runnable example below uses full fine-tuning. To train only `score`, use the following code on a freshly loaded model before creating the `Trainer`.

```python
# Freeze the pretrained decoder and inspect the trainable head.
gpt_model.base_model.requires_grad_(False)
for name, parameter in gpt_model.named_parameters():
    if parameter.requires_grad:
        print(name, parameter.numel())
```

The output should list `score.weight`. In this ticket experiment, freezing either the DistilBERT encoder or the GPT-2 decoder gave worse results than full fine-tuning. With a frozen backbone, the head can learn how to use existing features. Full fine-tuning also lets the encoder or decoder adapt those features to the ticket categories.


In [ ]:
import os
GPT_CHECKPOINT_DIR = "transformer/gpt-checkpoints"
os.makedirs(GPT_CHECKPOINT_DIR, exist_ok=True)
GPT_MAX_EPOCHS = 12  # Maximum passes; early stopping may finish sooner.
gpt_args = TrainingArguments(
    output_dir=GPT_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    num_train_epochs=GPT_MAX_EPOCHS, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("GPT-2 training device:", gpt_args.device)  # Show the device selected by Trainer.
gpt_trainer = Trainer(
    model=gpt_model, args=gpt_args,
    train_dataset=gpt_encoded_tickets["train"], eval_dataset=gpt_encoded_tickets["validation"],
    processing_class=gpt_tokenizer, data_collator=gpt_collator, compute_metrics=ticket_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
gpt_trainer.train()  # Update the decoder and task head using labelled training tickets.
print("Epochs completed:", gpt_trainer.state.epoch)
print("Best checkpoint:", gpt_trainer.state.best_model_checkpoint)
print("GPT-2 validation:", gpt_trainer.evaluate())


### 6.5 Compare the Three Methods

Use identical test tickets and labels for all three methods. Accuracy and macro F1 measure their predictions; confusion matrices and examples reveal different errors. The majority-class result gives a simple baseline.

The test set contains 64 fictional tickets, including the original 16 test examples. A pretrained Transformer is not guaranteed to beat the smaller NN, and a few changed predictions can substantially affect these scores.


In [ ]:
import matplotlib.pyplot as plt

after_ticket = ticket_trainer.predict(encoded_tickets["test"])
true_labels = after_ticket.label_ids
gpt_test_output = gpt_trainer.predict(gpt_encoded_tickets["test"])
gpt_predictions = gpt_test_output.predictions.argmax(axis=-1)
assert (gpt_test_output.label_ids == true_labels).all(), "Keep test rows aligned across models."
predicted_labels = after_ticket.predictions.argmax(axis=-1)
test_counts = count_vectorizer.transform(test_tickets["text"])
count_predictions = count_model.predict(test_counts)
majority_label = int(train_tickets["label"].mode().iloc[0])
majority_predictions = [majority_label] * len(true_labels)
comparison_rows = []
for name, predictions in [("Majority class", majority_predictions), ("Feature Engineering + MLP", count_predictions), ("BERT (DistilBERT)", predicted_labels), ("GPT (GPT-2)", gpt_predictions)]:
    comparison_rows.append({
        "model": name, "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, labels=list(id_to_label), average="macro", zero_division=0),
    })
print(pd.DataFrame(comparison_rows).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, name, predictions in [(axes[0], "Feature Engineering + MLP", count_predictions), (axes[1], "BERT (DistilBERT)", predicted_labels), (axes[2], "GPT (GPT-2)", gpt_predictions)]:
    print(name)
    print(classification_report(true_labels, predictions, labels=list(id_to_label), target_names=class_names, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        true_labels, predictions, labels=list(id_to_label), display_labels=class_names,
        xticks_rotation=30, cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
ticket_results = pd.DataFrame({
    "text": test_tickets["text"].tolist(),
    "actual": [id_to_label[int(label)] for label in true_labels],
    "word_counts_nn": [id_to_label[int(label)] for label in count_predictions],
    "distilbert": [id_to_label[int(label)] for label in predicted_labels],
    "gpt2": [id_to_label[int(label)] for label in gpt_predictions],
})
print("Tickets misclassified by any method:")
prediction_columns = ["word_counts_nn", "distilbert", "gpt2"]
errors = ticket_results[prediction_columns].ne(ticket_results["actual"], axis=0).any(axis=1)
print(ticket_results[errors].to_string(index=False))


### 6.6 Save, Reload and Route New Tickets

A softmax score is relative to the model's available classes. It is not a calibrated probability that the routing decision is correct. Out-of-scope tickets and ambiguous requests need a review process.

Save the count vocabulary together with the fitted scikit-learn MLP: changing the vocabulary order changes what each input column means. Save each Transformer tokenizer together with its model, including the GPT-2 padding configuration, so that inference uses the same input encoding.


In [ ]:
import os
import joblib
from transformers import pipeline

COUNT_SAVE_DIR = "transformer/saved-count-mlp"
os.makedirs(COUNT_SAVE_DIR, exist_ok=True)
# Keep the vocabulary, classifier and label names together for consistent inference.
joblib.dump({
    "vectorizer": count_vectorizer, "classifier": count_model, "class_names": class_names,
}, COUNT_SAVE_DIR + "/classifier.joblib")
saved_count = joblib.load(COUNT_SAVE_DIR + "/classifier.joblib")
reloaded_vectorizer = saved_count["vectorizer"]
reloaded_count_model = saved_count["classifier"]

TICKET_SAVE_DIR = "transformer/saved-ticket-classifier"
os.makedirs(TICKET_SAVE_DIR, exist_ok=True)
ticket_trainer.save_model(TICKET_SAVE_DIR)
ticket_tokenizer.save_pretrained(TICKET_SAVE_DIR)
ticket_classifier = pipeline("text-classification", model=TICKET_SAVE_DIR, tokenizer=TICKET_SAVE_DIR)
GPT_SAVE_DIR = "transformer/saved-gpt-classifier"
os.makedirs(GPT_SAVE_DIR, exist_ok=True)
gpt_trainer.save_model(GPT_SAVE_DIR)
gpt_tokenizer.save_pretrained(GPT_SAVE_DIR)
gpt_classifier = pipeline("text-classification", model=GPT_SAVE_DIR, tokenizer=GPT_SAVE_DIR)

new_tickets = [
    "My laptop cannot connect to campus Wi-Fi.",
    "I dropped my monitor and the display is broken.",
    "The program crashes, but only when I connect to Wi-Fi.",
]
new_counts = reloaded_vectorizer.transform(new_tickets)
count_classes = reloaded_count_model.predict(new_counts)
bert_results = ticket_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
gpt_results = gpt_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
for text, count_class, bert_result, gpt_result in zip(new_tickets, count_classes, bert_results, gpt_results):
    print(text)
    print("Feature Engineering + MLP:", saved_count["class_names"][count_class])
    print("BERT (DistilBERT):", bert_result, "| GPT (GPT-2):", gpt_result)
